## Setup

In [1]:
import importlib
from pathlib import Path
import sys
import polars as pl

pl.Config.set_tbl_rows(25)

REPO_DIR = Path('.').resolve().parent
DEV_DATA_DIR = REPO_DIR / 'trio_dev_data'

sys.path.insert(0, str(REPO_DIR / 'src'))
sys.path.insert(0, str(REPO_DIR / 'src' / 'util'))

In [2]:
KID_ID = 'NA12878'
DAD_ID = 'NA12891'
MOM_ID = 'NA12892'

METH_BASE_DIR = DEV_DATA_DIR / 'output' / 'dna-methylation'

# Unphased (combined) methylation BEDs from aligned_bam_to_cpg_scores
METH_COUNT_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.count.pedmec-phased' # output dir of aligned_bam_to_cpg_scores (containing count-based unphased meth)
METH_MODEL_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.model.pedmec-phased' # output dir of aligned_bam_to_cpg_scores (containing model-based unphased meth)

def combined_meth_paths(uid):
    """Return (count_combined, model_combined) paths for a sample."""
    return (
        str(METH_COUNT_DIR / f'{uid}.GRCh38.haplotagged.combined.bed.gz'), # unphased count-based meth from aligned_bam_to_cpg_scores
        str(METH_MODEL_DIR / f'{uid}.GRCh38.haplotagged.combined.bed.gz'), # unphased model-based meth from aligned_bam_to_cpg_scores
    )

KID_METH_COUNT_COMBINED, KID_METH_MODEL_COMBINED = combined_meth_paths(KID_ID)
DAD_METH_COUNT_COMBINED, DAD_METH_MODEL_COMBINED = combined_meth_paths(DAD_ID)
MOM_METH_COUNT_COMBINED, MOM_METH_MODEL_COMBINED = combined_meth_paths(MOM_ID)

# Parent-phased methylation BED
PARENT_PHASED_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.parent-phased' # output dir of phase_meth_to_parent_haps.py
BED_METH_PARENT_PHASED = str(PARENT_PHASED_DIR / 'trio.dna-methylation.bed') # parent-phased methylation levels from phase_meth_to_parent_haps.py

# Heterozygous sites at which bit-vectors are mismatched, from phase_meth_to_parent_haps.py
BED_HET_SITE_MISMATCHES_PAT = str(PARENT_PHASED_DIR / f'{KID_ID}.bit-vector-sites-mismatches.paternal.bed')
BED_HET_SITE_MISMATCHES_MAT = str(PARENT_PHASED_DIR / f'{KID_ID}.bit-vector-sites-mismatches.maternal.bed')

# All CpG sites in the reference genome
OUTPUT_DIR = METH_BASE_DIR / 'CEPH1463.GRCh38.hifi.parent-phased.all-cpgs'
BED_ALL_CPGS_IN_REFERENCE = str(OUTPUT_DIR / 'all_cpg_sites_in_reference.bed') # output of write_all_cpgs_in_reference.py

## Get all CpG sites in reference genome

In [3]:
import util
importlib.reload(util)
from util import read_all_cpgs_in_reference

DF_ALL_CPGS_IN_REFERENCE = read_all_cpgs_in_reference(BED_ALL_CPGS_IN_REFERENCE)
DF_ALL_CPGS_IN_REFERENCE

chrom,start,end
str,i64,i64
"""chr1""",10468,10469
"""chr1""",10470,10471
"""chr1""",10483,10484
"""chr1""",10488,10489
"""chr1""",10492,10493
"""chr1""",10496,10497
"""chr1""",10524,10525
"""chr1""",10541,10542
"""chr1""",10562,10563


## Read in unphased DNA methylation at CpG sites for kid, dad, and mom

In [4]:
import expand_to_all_cpgs_trio 
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import read_meth_unphased_trio

DF_METH_UNPHASED = read_meth_unphased_trio(
    bed_meth_count_kid=KID_METH_COUNT_COMBINED,
    bed_meth_model_kid=KID_METH_MODEL_COMBINED,
    bed_meth_count_dad=DAD_METH_COUNT_COMBINED,
    bed_meth_model_dad=DAD_METH_MODEL_COMBINED,
    bed_meth_count_mom=MOM_METH_COUNT_COMBINED,
    bed_meth_model_mom=MOM_METH_MODEL_COMBINED,
)
DF_METH_UNPHASED

chrom,start,end,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64
"""chr1""",986903,986904,10,0.6,0.666,11,0.455,0.49,21,0.476,0.538
"""chr1""",986966,986967,10,0.8,0.942,11,0.818,0.897,21,0.524,0.772
"""chr1""",987029,987030,10,0.8,0.941,12,0.667,0.817,21,0.476,0.523
"""chr1""",987116,987117,10,0.5,0.878,13,0.462,0.536,21,0.667,0.842
"""chr1""",987154,987155,11,0.909,0.944,13,0.769,0.864,21,0.619,0.9
"""chr1""",987343,987344,11,0.545,0.843,13,0.846,0.888,22,0.591,0.923
"""chr1""",987366,987367,11,0.909,0.945,13,0.538,0.526,22,0.818,0.946
"""chr1""",987425,987426,12,0.75,0.928,13,0.846,0.923,21,0.905,0.962
"""chr1""",987427,987428,12,0.583,0.91,13,0.846,0.935,22,0.864,0.933


## Read in parent-phased DNA methylation at CpG sites

In [5]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import read_meth_parent_phased

DF_METH_PARENT_PHASED = read_meth_parent_phased(BED_METH_PARENT_PHASED)
DF_METH_PARENT_PHASED

chrom,start,end,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom
str,i64,i64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64
"""chr1""",990862,990863,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990869,990870,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990914,990915,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990924,990925,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990948,990949,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",990956,990957,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",991280,991281,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",991431,991432,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",991460,991461,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


## Expand the dataframe of parent-phased methylation levels to include all CpG sites in reference and sample genome, and unphased methylation levels (where available)

In [6]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import expand_meth_to_all_cpgs

DF_METH_PARENT_PHASED_ALL_CPGS = expand_meth_to_all_cpgs(DF_ALL_CPGS_IN_REFERENCE, DF_METH_UNPHASED, DF_METH_PARENT_PHASED)
DF_METH_PARENT_PHASED_ALL_CPGS

chrom,start,end,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64
"""chr1""",10468,10469,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10470,10471,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10483,10484,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10488,10489,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10492,10493,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10496,10497,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10524,10525,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10541,10542,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
"""chr1""",10562,10563,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


## Add proximity of each CpG site to heterozygous sites at which bit-vectors are mismatched

In [7]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import compute_proximity_to_mismatched_heterozygous_sites

DF_METH_PARENT_PHASED_ALL_CPGS = compute_proximity_to_mismatched_heterozygous_sites(
    DF_METH_PARENT_PHASED_ALL_CPGS,
    BED_HET_SITE_MISMATCHES_PAT,
    BED_HET_SITE_MISMATCHES_MAT,
)
DF_METH_PARENT_PHASED_ALL_CPGS

chrom,start,end,total_read_count_kid,methylation_level_kid_count,methylation_level_kid_model,total_read_count_dad,methylation_level_dad_count,methylation_level_dad_model,total_read_count_mom,methylation_level_mom_count,methylation_level_mom_model,total_read_count_kid_pat,methylation_level_kid_pat_count,total_read_count_kid_mat,methylation_level_kid_mat_count,methylation_level_kid_pat_model,methylation_level_kid_mat_model,total_read_count_dad_A,methylation_level_dad_A_count,total_read_count_dad_B,methylation_level_dad_B_count,methylation_level_dad_A_model,methylation_level_dad_B_model,total_read_count_mom_C,methylation_level_mom_C_count,total_read_count_mom_D,methylation_level_mom_D_count,methylation_level_mom_C_model,methylation_level_mom_D_model,start_hap_map_block_pat,end_hap_map_block_pat,paternal_haplotype,paternal_concordance,num_het_SNVs_in_dad,start_hap_map_block_mat,end_hap_map_block_mat,maternal_haplotype,maternal_concordance,num_het_SNVs_in_mom,is_within_50bp_of_mismatch_site_pat,is_within_50bp_of_mismatch_site_mat
str,i64,i64,i64,f64,f64,i64,f64,f64,i64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,f64,i64,f64,f64,f64,i64,i64,str,f64,i64,i64,i64,str,f64,i64,bool,bool
"""chr1""",10468,10469,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10470,10471,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10483,10484,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10488,10489,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10492,10493,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10496,10497,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10524,10525,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10541,10542,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false
"""chr1""",10562,10563,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,false,false


## QC Statistics 

In [8]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import compute_fraction_of_cpgs_that_are_close_to_mismatches

compute_fraction_of_cpgs_that_are_close_to_mismatches(DF_METH_PARENT_PHASED_ALL_CPGS)

Percentage of CpG sites (in reference and sample genome, and on phasable chroms) that are within 50bp of a paternal heterozygous mismatch site: 0.000%
Percentage of CpG sites (in reference and sample genome, and on phasable chroms) that are within 50bp of a maternal heterozygous mismatch site: 0.038%


In [9]:
importlib.reload(expand_to_all_cpgs_trio)
from expand_to_all_cpgs_trio import compute_methylation_coverage_qc

compute_methylation_coverage_qc(DF_METH_PARENT_PHASED_ALL_CPGS)

Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to kid_pat haplotype: 1.41%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to kid_mat haplotype: 1.38%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to at least one parental haplotype (kid): 1.45%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based methylation is phased to both parental haplotypes (kid): 1.33%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which count-based unphased methylation is reported for kid: 1.59%
Percentage of CpG sites (in reference and sample genomes, and on phasable chroms) at which model-based methylation is phased to kid_pat haplotype: 1.41%
Percentage of CpG sites (in reference and sample ge